# NYC 311 — Exploratory Analysis

Lightweight EDA on the cleaned 12-month dataset (Jul 2025 – Jun 2026).
Run `scripts/01_fetch_311_data.py` and `scripts/02_clean_transform.py` first.

Questions explored here feed the dashboard design:
1. Volume by borough / ZIP
2. Resolution-time distribution (why we use medians)
3. High-volume vs slow-resolution complaint types
4. Agency workload vs speed
5. Seasonal / weekly / hourly demand patterns

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_parquet(ROOT / 'data' / 'processed' / '311_clean.parquet')
print(f'{len(df):,} rows | {df.created_date.min():%Y-%m-%d} to {df.created_date.max():%Y-%m-%d}')
df.head(3)

## 1. Volume by borough

In [ ]:
borough_counts = df['borough'].value_counts()
display(borough_counts)
borough_counts.drop('Unspecified', errors='ignore').plot.barh(
    title='311 requests by borough (12 months)', figsize=(8, 4))
plt.gca().invert_yaxis(); plt.tight_layout()

## 2. Resolution-time distribution — why medians

The long right tail below is the reason every headline metric in this project
uses the **median**, not the mean.

In [ ]:
closed = df.loc[df['is_closed'], 'resolution_days'].dropna()
stats = closed.describe(percentiles=[0.5, 0.75, 0.9, 0.99])
print(stats.round(2))
print(f'\nMean {closed.mean():.1f} vs median {closed.median():.2f} days')

closed.clip(upper=30).plot.hist(bins=60, figsize=(8, 3.5),
    title='Resolution days, closed requests (clipped at 30 for display)')
plt.axvline(closed.median(), color='red', ls='--', label=f'median {closed.median():.2f}d')
plt.legend(); plt.tight_layout()

## 3. Complaint types: volume vs speed

In [ ]:
ct = (df.groupby('complaint_type', observed=True)
        .agg(request_count=('unique_key', 'size'),
             median_res=('resolution_days', 'median'),
             open_rate=('is_closed', lambda s: 1 - s.mean()))
        .query('request_count >= 200'))

print('Top 10 by volume:')
display(ct.nlargest(10, 'request_count').round(2))

ax = ct.plot.scatter(x='request_count', y='median_res', logx=True, alpha=0.5,
                     figsize=(8, 5), title='Complaint types: volume vs median resolution days')
ax.axhline(ct['median_res'].median(), color='gray', ls=':')
ax.axvline(ct['request_count'].median(), color='gray', ls=':')
for name, row in ct.nlargest(5, 'request_count').iterrows():
    ax.annotate(name, (row['request_count'], row['median_res']), fontsize=8)
plt.tight_layout()

## 4. Agency workload vs speed

In [ ]:
ag = (df.groupby('agency', observed=True)
        .agg(request_count=('unique_key', 'size'),
             median_res=('resolution_days', 'median'),
             open_rate=('is_closed', lambda s: 1 - s.mean()))
        .nlargest(12, 'request_count'))
display(ag.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
ag['request_count'].plot.barh(ax=axes[0], title='Requests')
ag['median_res'].plot.barh(ax=axes[1], color='darkorange', title='Median resolution days')
axes[0].invert_yaxis(); plt.tight_layout()

## 5. Time patterns

In [ ]:
monthly = df.groupby('created_month', observed=True).size()
monthly.plot(title='Monthly request volume', figsize=(9, 3.5), marker='o')
plt.tight_layout()
print(f'Peak month: {monthly.idxmax()} ({monthly.max():,}) | Low: {monthly.idxmin()} ({monthly.min():,})')

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = (df.groupby(['created_day_of_week', 'created_hour'], observed=True)
           .size().unstack().reindex(day_order))

fig, ax = plt.subplots(figsize=(11, 3.5))
im = ax.imshow(pivot, aspect='auto', cmap='OrRd')
ax.set_yticks(range(7), day_order)
ax.set_xticks(range(0, 24, 2), range(0, 24, 2))
ax.set_title('Request volume by weekday and hour')
fig.colorbar(im, shrink=0.8); plt.tight_layout()

## Takeaways

Findings from these views (with exact numbers) are written up in
`docs/insights_and_recommendations.md` and drive the dashboard storyboard.